In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error

In [3]:
# ── 1) LOAD DATA ────────────────────────────────────────────
DATA_DIR = "data"

books_df = pd.read_csv(
    os.path.join(DATA_DIR, "Books.csv"),
    low_memory=False      # handle mixed-type columns quietly
)
ratings = pd.read_csv(os.path.join(DATA_DIR, "Ratings.csv"))
users_df = pd.read_csv(os.path.join(DATA_DIR, "Users.csv"))

print(f"Books:   {books_df.shape[0]:,} rows × {books_df.shape[1]} cols")
print(f"Ratings: {ratings.shape[0]:,} rows × {ratings.shape[1]} cols")
print(f"Users:   {users_df.shape[0]:,} rows × {users_df.shape[1]} cols")

Books:   271,360 rows × 8 cols
Ratings: 1,149,780 rows × 3 cols
Users:   278,858 rows × 3 cols


In [4]:
# ── 2) FILTER OUT IMPLICIT “0” RATINGS ───────────────────────
ratings = ratings[ratings["Book-Rating"] > 0]
print(f"Explicit ratings only: {ratings.shape[0]:,} rows")

Explicit ratings only: 433,671 rows


In [5]:
# ── 3) INTERACTION-LEVEL SPLIT ──────────────────────────────
def split_interactions(df, test_frac=0.25, random_state=42):
    test_df  = df.sample(frac=test_frac, random_state=random_state)
    train_df = df.drop(test_df.index)

    # move any “cold” users/items back into train
    mask_cold = (
        ~test_df["User-ID"].isin(train_df["User-ID"]) |
        ~test_df["ISBN"   ].isin(train_df["ISBN"   ])
    )
    while mask_cold.any():
        to_move  = test_df[mask_cold]
        train_df = pd.concat([train_df, to_move], ignore_index=True)
        test_df  = test_df[~mask_cold].reset_index(drop=True)
        mask_cold = (
            ~test_df["User-ID"].isin(train_df["User-ID"]) |
            ~test_df["ISBN"   ].isin(train_df["ISBN"   ])
        )

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, test_df = split_interactions(ratings, test_frac=0.25, random_state=42)
print(f"Train pairs: {len(train_df):,}    Test pairs: {len(test_df):,}")

Train pairs: 370,443    Test pairs: 63,228


In [6]:
# ── 4) BUILD SPARSE USER–ITEM MATRIX ────────────────────────
user_ids = train_df["User-ID"].unique()
item_ids = train_df["ISBN"].unique()
u2i = {u:i for i,u in enumerate(user_ids)}
i2i = {i:j for j,i in enumerate(item_ids)}

rows = train_df["User-ID"].map(u2i).to_numpy()
cols = train_df["ISBN"   ].map(i2i).to_numpy()
data = train_df["Book-Rating"].to_numpy()

R = csr_matrix((data, (rows, cols)),
               shape=(len(user_ids), len(item_ids)))

In [7]:
# ── 5) FIT USER–USER k-NN (COSINE) ──────────────────────────
model = NearestNeighbors(metric="cosine", algorithm="brute")
model.fit(R)

NearestNeighbors(algorithm='brute', metric='cosine')

In [8]:
# ── 6) PRECOMPUTE MEANS FOR FALLBACKS ───────────────────────
user_means  = train_df.groupby("User-ID")["Book-Rating"].mean().to_dict()
item_means  = train_df.groupby("ISBN"   )["Book-Rating"].mean().to_dict()
global_mean = train_df["Book-Rating"].mean()

In [17]:
# ── 7) PREDICTION WITH SAFETY CHECKS ────────────────────────
def predict_rating(user, item, k=10):
    # fallback for unknown user/item
 
    if user not in u2i or item not in i2i:
        print(f"Warning: {user} or {item} not in training data.")
        return user_means.get(user,
               item_means.get(item,
               global_mean))

    u_idx = u2i[user]
    i_idx = i2i[item]

    # find k+1 neighbors (includes self)
    dists, nbrs = model.kneighbors(R[u_idx], n_neighbors=k+1)
    sims = 1 - dists.flatten()
    nbrs = nbrs.flatten()

    # drop self
    mask = nbrs != u_idx
    sims = sims[mask][:k]
    nbrs = nbrs[mask][:k]

    # neighbors' ratings for this item
    r_i = R[nbrs, i_idx].toarray().flatten()
    valid = r_i > 0

    # if at least one neighbor rated it and sims sum >0
    if valid.any() and sims[valid].sum() > 0:
        return np.dot(sims[valid], r_i[valid]) / sims[valid].sum()
    print(f"Warning: {user} rated {item} but no neighbors rated it.")
    # otherwise fallback to user/item/global mean
    return user_means.get(user,
           item_means.get(item,
           global_mean))

In [24]:
print(predict_rating(16,1559703237))

9.0


In [12]:
# ── 8) EVALUATION LOOP: MAE vs k ─────────────────────────────
def evaluate(test_df, ks=[5,10,15,20,50,100]):
    for k in ks:
        preds, acts = [], []
        for _, row in test_df.iterrows():
            u, isbn, true_r = row["User-ID"], row["ISBN"], row["Book-Rating"]
            preds.append(predict_rating(u, isbn, k))
            acts.append(true_r)
        mae = mean_absolute_error(acts, preds)
        print(f"k = {k:<3} → MAE = {mae:.4f}")

print("\nEvaluating on test set:")
evaluate(test_df)

ks = [5, 10, 15, 20, 50, 100]
errors = [mae_results[k] for k in ks]

# ── 9) PLOT MAE vs k ─────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.plot(ks, errors, marker='o')
plt.title("User–User CF: MAE vs. Neighborhood Size (k)")
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Mean Absolute Error")
plt.xticks(ks)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


Evaluating on test set:


KeyboardInterrupt: 